In [4]:
# ====================================
# 1. 라이브러리 import
# ====================================
import os
import glob
import random
import shutil # 폴더 삭제, 생성 등 파일시스템 작업에 사용

# 데이터 처리 및 시각화 라이브러리
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 오디오 라이브러리
import librosa # 오디오 로딩
import librosa.display

# 이미지 처리 라이브러리
from PIL import Image

# 분석 라이브러리
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# ============================================
# 2. 기본 경로 및 전역 하이퍼 파라미터 설정
# ============================================

BASE_DIR = os.getcwd() # root 디렉토리 설정 (workspace)
AUDIO_DIR = os.path.join(BASE_DIR, "audio") # 오디오 파일이 저장된 루트 폴더 지정
SPCE_DIR = os.path.join(BASE_DIR, "spectrogram_data") # 스펙트로그램으로 변환한 이미지 저장할 폴더 지정
SPLIT_DIR = os.path.join(BASE_DIR, "spectrogram_split") # 학습 데이터가 저장될 폴더 지정 (ai 학습용으로 train/test/val 구조로 저장할 예정)

# 최종 학습 모델 결과를 저장할 경로 및 파일명 지정
BEST_MODEL_PATH = os.path.join(BASE_DIR, "best_instrument_cnn_model.keras") 
#신규 예측 시 임시로 스펙트로그램 이미지를 저장할 경로 및 파일명 지정
TEMP_PREDICT_IMAGE_PATH = os.path.join(BASE_DIR, "temp_predict_spectrogram.png") 

# 분류할 클래스는 6개 (폴더명과 일치하도록 정의하기)
CLASSES = ['cello','flute','sax','oboe','trumpet','viola']

# CNN 입력 이미지 높이
IMG_HEIGHT = 224
# CNN 입력 이미지 너비
IMG_WIDTH = 224

# Signal processing 파라미터 설정 (전문가 영역/깊게 알기 전 감잡기)
SR = 44100 # 오디오 샘플링 주파수
DURATION = 1.5 # 오디오 길이 (초단위)
SAMPLES_PER_TRACK = SR*DURATION # 1개 오디오 샘플당 기대되는 총 샘플 수
N_FFT = 2048 # 멜 스펙트로그램 생성 시 사용할 FFT 윈도우 길이 (시그널 데이터도 시계열 형식으로 들어올때 윈도우 구간으로 잘라서 분석한다. 얼마나 자를지 지정한 것이다.)
HOP_LENGTH = 1024
N_MELS = 126

BATCH_SIZE = 16
EPOCHS = 20